# 04 — Data Preprocessing Pipeline

**Goal**: Clean and merge all source datasets into a single `master_movies.parquet` feature table, plus time-split `ratings_train / ratings_val / ratings_test` Parquet files ready for model training.

**Pipeline steps:**
1. Load & clean `movies_metadata.csv` → `movies_clean.parquet`
2. Load & clean `credits.csv` → `credits_clean.parquet`
3. Load & clean `keywords.csv` → `keywords_clean.parquet`
4. Load & filter IMDB `title.basics` + `title.ratings` → `imdb_enrichment.parquet`
5. Execute join chain → `master_movies.parquet`
6. Engineer features: `overview_clean`, `content_soup`, `bayesian_score`
7. Time-split `ratings.csv` → `ratings_train / _val / _test.parquet`
8. Validate all outputs

---

In [1]:
import ast
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = Path(r'e:\Projects\Movies Recommendation Engine')
MOVIELENS   = ROOT / 'data' / 'movielens'
IMDB        = ROOT / 'data' / 'imdb'
PROCESSED   = ROOT / 'data_science' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

print('Paths OK')
print(f'  MovieLens : {MOVIELENS}')
print(f'  IMDB      : {IMDB}')
print(f'  Processed : {PROCESSED}')

Paths OK
  MovieLens : e:\Projects\Movies Recommendation Engine\data\movielens
  IMDB      : e:\Projects\Movies Recommendation Engine\data\imdb
  Processed : e:\Projects\Movies Recommendation Engine\data_science\processed


---
## Step 1 — Clean `movies_metadata.csv`

Key operations:
- Drop 3 rows where `id` is a URL string (non-numeric)
- Cast `id` → int64 (becomes `tmdb_id`)
- Parse `genres` JSON list → plain Python list of genre names
- Extract `release_year` from `release_date`
- Coerce `runtime` and `vote_*` columns to numeric
- Drop 12 columns we don't need

In [2]:
DROP_TMDB = [
    'belongs_to_collection', 'homepage', 'tagline', 'video', 'adult',
    'status', 'poster_path', 'backdrop_path', 'imdb_id',
    'production_companies', 'production_countries', 'spoken_languages',
]

def parse_json_list(val, key='name'):
    """Safely parse a stringified JSON list and return a list of values for `key`."""
    try:
        items = ast.literal_eval(val)
        return [d[key] for d in items if isinstance(d, dict) and key in d]
    except Exception:
        return []

print('Loading movies_metadata.csv …')
movies_raw = pd.read_csv(MOVIELENS / 'movies_metadata.csv', low_memory=False)
print(f'  Raw shape: {movies_raw.shape}')

# Drop rows where id is not numeric (3 URL rows)
movies_raw = movies_raw[pd.to_numeric(movies_raw['id'], errors='coerce').notna()].copy()
movies_raw['tmdb_id'] = movies_raw['id'].astype(int)

# Parse genre JSON → list of names
movies_raw['genre_list'] = movies_raw['genres'].apply(lambda x: parse_json_list(x, 'name'))

# Extract release_year
movies_raw['release_year'] = pd.to_datetime(
    movies_raw['release_date'], errors='coerce'
).dt.year.astype('Int64')

# Coerce numeric columns
for col in ['runtime', 'vote_average', 'vote_count', 'popularity']:
    movies_raw[col] = pd.to_numeric(movies_raw[col], errors='coerce')

# Rename & select
movies_clean = (
    movies_raw
    .rename(columns={
        'vote_average'  : 'tmdb_score',
        'vote_count'    : 'tmdb_votes',
        'popularity'    : 'tmdb_popularity',
        'runtime'       : 'runtime_min',
        'original_language': 'language',
    })
    .drop(columns=DROP_TMDB, errors='ignore')
    [[  # explicit keep order
        'tmdb_id', 'title', 'original_title', 'overview',
        'genre_list', 'release_year', 'language',
        'runtime_min', 'tmdb_score', 'tmdb_votes', 'tmdb_popularity',
    ]]
    .drop_duplicates(subset='tmdb_id')
    .reset_index(drop=True)
)

print(f'  Cleaned shape: {movies_clean.shape}')
print(f'  Null overview : {movies_clean["overview"].isna().sum():,}')
movies_clean.head(3)

Loading movies_metadata.csv …
  Raw shape: (45466, 24)
  Cleaned shape: (45433, 11)
  Null overview : 954


,tmdb_id,title,original_title,overview,genre_list,release_year,language,runtime_min,tmdb_score,tmdb_votes,tmdb_popularity
0,862,Toy Story,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[Animation, Comedy, Family]",1995,en,81.0,7.7,5415.0,21.946943
1,8844,Jumanji,Jumanji,When siblings Judy and Peter discover an encha...,"[Adventure, Fantasy, Family]",1995,en,104.0,6.9,2413.0,17.015539
2,15602,Grumpier Old Men,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[Romance, Comedy]",1995,en,101.0,6.5,92.0,11.712900


---
## Step 2 — Clean `credits.csv`

Key operations:
- Parse `crew` JSON → extract first `Director` entry
- Parse `cast` JSON → extract top-3 cast names (by order)
- Normalise names: lowercase, remove spaces (for content soup)

In [3]:
def extract_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str)
        for member in crew:
            if isinstance(member, dict) and member.get('job') == 'Director':
                return member.get('name', '')
    except Exception:
        pass
    return ''

def extract_top_cast(cast_str, n=3):
    try:
        cast = ast.literal_eval(cast_str)
        # sorted by 'order' ascending, take first n
        cast_sorted = sorted(
            [m for m in cast if isinstance(m, dict)],
            key=lambda x: x.get('order', 999)
        )
        return [m['name'] for m in cast_sorted[:n] if 'name' in m]
    except Exception:
        return []

def normalise_name(name):
    """Lowercase and strip spaces so 'John Ford' → 'johnford' (for soup)."""
    if not name or not isinstance(name, str):
        return ''
    return re.sub(r'\s+', '', name.lower().strip())

print('Loading credits.csv …')
credits_raw = pd.read_csv(MOVIELENS / 'credits.csv')
print(f'  Raw shape: {credits_raw.shape}')

credits_raw['id'] = pd.to_numeric(credits_raw['id'], errors='coerce')
credits_raw = credits_raw.dropna(subset=['id']).copy()
credits_raw['id'] = credits_raw['id'].astype(int)

credits_raw['director']  = credits_raw['crew'].apply(extract_director)
credits_raw['cast_top3'] = credits_raw['cast'].apply(extract_top_cast)

credits_clean = (
    credits_raw[['id', 'director', 'cast_top3']]
    .rename(columns={'id': 'tmdb_id'})
    .drop_duplicates(subset='tmdb_id')
    .reset_index(drop=True)
)

print(f'  Cleaned shape: {credits_clean.shape}')
print(f'  Missing director: {(credits_clean["director"] == "").sum():,}')
credits_clean.head(3)

Loading credits.csv …
  Raw shape: (45476, 3)
  Cleaned shape: (45432, 3)
  Missing director: 887


,tmdb_id,director,cast_top3
0,862,John Lasseter,"[Tom Hanks, Tim Allen, Don Rickles]"
1,8844,Joe Johnston,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]"
2,15602,Howard Deutch,"[Walter Matthau, Jack Lemmon, Ann-Margret]"


---
## Step 3 — Clean `keywords.csv`

Key operations:
- Parse `keywords` JSON → list of keyword strings
- Remove noise keywords: `duringcreditsstinger`, `aftercreditsstinger`, `woman director`, `independent film`, `based on novel`
- Normalise keywords: lowercase, strip spaces (for soup)

In [4]:
NOISE_KEYWORDS = {
    'duringcreditsstinger', 'aftercreditsstinger',
    'woman director', 'independent film', 'based on novel',
    'based on novel or book', 'based on true story',
}

def clean_keywords(kw_str):
    try:
        kws = ast.literal_eval(kw_str)
        names = [d['name'].lower().strip() for d in kws if isinstance(d, dict) and 'name' in d]
        return [k for k in names if k not in NOISE_KEYWORDS]
    except Exception:
        return []

print('Loading keywords.csv …')
kw_raw = pd.read_csv(MOVIELENS / 'keywords.csv')
print(f'  Raw shape: {kw_raw.shape}')

kw_raw['id'] = pd.to_numeric(kw_raw['id'], errors='coerce')
kw_raw = kw_raw.dropna(subset=['id']).copy()
kw_raw['id'] = kw_raw['id'].astype(int)

keywords_clean = (
    kw_raw.assign(keywords_clean=kw_raw['keywords'].apply(clean_keywords))
    [['id', 'keywords_clean']]
    .rename(columns={'id': 'tmdb_id'})
    .drop_duplicates(subset='tmdb_id')
    .reset_index(drop=True)
)

print(f'  Cleaned shape: {keywords_clean.shape}')
print(f'  Median keyword count: {keywords_clean["keywords_clean"].apply(len).median()}')
keywords_clean.head(3)

Loading keywords.csv …
  Raw shape: (46419, 2)
  Cleaned shape: (45432, 2)
  Median keyword count: 2.0


,tmdb_id,keywords_clean
0,862,"[jealousy, toy, boy, friendship, friends, riva..."
1,8844,"[board game, disappearance, based on children'..."
2,15602,"[fishing, best friend, old men]"


---
## Step 4 — Load & Filter IMDB Files

Strategy:
1. Load `links.csv` → build the set of valid IMDB `tconst` IDs (format: `tt0114709`)
2. Read `title.basics.tsv` (runtime, genres) — filter to valid set, keep movies only
3. Read `title.ratings.tsv` (averageRating, numVotes) — filter to valid set
4. Join basics + ratings on `tconst`

IMDB null sentinel is `\N` — handled via `na_values=r'\N'`.

In [5]:
print('Loading links.csv to build tconst filter set …')
links = pd.read_csv(MOVIELENS / 'links.csv', dtype={'imdbId': str})
links['tmdb_id']     = pd.to_numeric(links['tmdbId'], errors='coerce').astype('Int64')
links['imdb_tconst'] = 'tt' + links['imdbId'].str.zfill(7)
links = links.dropna(subset=['tmdb_id'])[['tmdb_id', 'imdb_tconst']].drop_duplicates()

valid_tconst = set(links['imdb_tconst'])
print(f'  Valid tconst IDs: {len(valid_tconst):,}')

# ── title.basics ──────────────────────────────────────────────────────────────
print('Loading title.basics.tsv (movies only) …')
basics_chunks = pd.read_csv(
    IMDB / 'title.basics.tsv',
    sep='\t', na_values=r'\N',
    usecols=['tconst', 'titleType', 'runtimeMinutes'],
    dtype={'tconst': str, 'titleType': str, 'runtimeMinutes': str},
    chunksize=200_000,
)

basics_list = []
for chunk in basics_chunks:
    filtered = chunk[
        chunk['tconst'].isin(valid_tconst) &
        (chunk['titleType'] == 'movie')
    ]
    if not filtered.empty:
        basics_list.append(filtered)

basics = pd.concat(basics_list, ignore_index=True)
basics['runtimeMinutes'] = pd.to_numeric(basics['runtimeMinutes'], errors='coerce')
print(f'  basics shape: {basics.shape}')

# ── title.ratings ─────────────────────────────────────────────────────────────
print('Loading title.ratings.tsv …')
ratings_imdb = pd.read_csv(
    IMDB / 'title.ratings.tsv',
    sep='\t', na_values=r'\N',
    dtype={'tconst': str},
)
ratings_imdb = ratings_imdb[ratings_imdb['tconst'].isin(valid_tconst)]
print(f'  ratings shape: {ratings_imdb.shape}')

# ── Join basics + ratings ─────────────────────────────────────────────────────
imdb_enrichment = basics.merge(ratings_imdb, on='tconst', how='left')
imdb_enrichment = imdb_enrichment.rename(columns={
    'tconst'          : 'imdb_tconst',
    'runtimeMinutes'  : 'imdb_runtime_min',
    'averageRating'   : 'imdb_score',
    'numVotes'        : 'imdb_votes',
})[['imdb_tconst', 'imdb_runtime_min', 'imdb_score', 'imdb_votes']]

print(f'  imdb_enrichment shape: {imdb_enrichment.shape}')
imdb_enrichment.head(3)

Loading links.csv to build tconst filter set …
  Valid tconst IDs: 45,624
Loading title.basics.tsv (movies only) …
  basics shape: (39176, 3)
Loading title.ratings.tsv …
  ratings shape: (45527, 3)
  imdb_enrichment shape: (39176, 4)


,imdb_tconst,imdb_runtime_min,imdb_score,imdb_votes
0,tt0000574,70.0,6.0,1064.0
1,tt0002101,100.0,5.2,668.0
2,tt0002130,71.0,7.0,4067.0


---
## Step 5 — Execute Join Chain

```
movies_clean
  LEFT JOIN credits_clean     ON tmdb_id
  LEFT JOIN keywords_clean    ON tmdb_id
  LEFT JOIN links             ON tmdb_id
  LEFT JOIN imdb_enrichment   ON imdb_tconst
```

Using LEFT JOINs throughout so we never lose a movie; IMDB enrichment will be NaN for unmatched rows.

In [6]:
print('Executing join chain …')

master = (
    movies_clean
    .merge(credits_clean,   on='tmdb_id', how='left')
    .merge(keywords_clean,  on='tmdb_id', how='left')
    .merge(links[['tmdb_id', 'imdb_tconst']], on='tmdb_id', how='left')
    .merge(imdb_enrichment, on='imdb_tconst', how='left')
)

print(f'  master shape after joins: {master.shape}')
print()
print('Null counts per column:')
print(master.isnull().sum().to_string())

Executing join chain …
  master shape after joins: (45463, 18)

Null counts per column:
tmdb_id                0
title                  3
original_title         0
overview             954
genre_list             0
release_year          87
language              11
runtime_min          260
tmdb_score             3
tmdb_votes             3
tmdb_popularity        3
director               1
cast_top3              1
keywords_clean         1
imdb_tconst            0
imdb_runtime_min    6377
imdb_score          6351
imdb_votes          6351


---
## Step 6 — Feature Engineering

### 6a — `overview_clean`
- Strip whitespace, collapse multiple spaces, fill missing with empty string.

### 6b — `bayesian_score`
Formula: $\hat{R} = \dfrac{C \cdot m + R \cdot v}{m + v}$

Where:
- $C$ = global mean of `imdb_score` (fallback: `tmdb_score`)
- $m$ = minimum vote threshold (100 votes)
- $R$ = movie's own score
- $v$ = movie's vote count

### 6c — `content_soup`
Concatenated string of normalised tokens used for TF-IDF:
```
genres + director + cast_top3 + keywords + overview
```
Names have whitespace removed (e.g. `"John Ford"` → `"johnford"`) so they are treated as single tokens.

In [7]:
# ── 6a: overview_clean ────────────────────────────────────────────────────────
master['overview_clean'] = (
    master['overview']
    .fillna('')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# ── 6b: bayesian_score ────────────────────────────────────────────────────────
VOTE_THRESHOLD = 100  # minimum votes for Bayesian estimate

# Derive a unified score and vote count: prefer IMDB, fall back to TMDB
master['_score'] = master['imdb_score'].combine_first(master['tmdb_score'])
master['_votes'] = master['imdb_votes'].combine_first(master['tmdb_votes'])

C = master['_score'].mean()   # global mean score (~6.96)
m = VOTE_THRESHOLD

master['bayesian_score'] = (
    (C * m + master['_score'] * master['_votes'])
    / (m + master['_votes'])
).round(4)

print(f'Global mean score C = {C:.4f}')
print(f'Bayesian score — min: {master["bayesian_score"].min():.4f}, '
      f'max: {master["bayesian_score"].max():.4f}, '
      f'mean: {master["bayesian_score"].mean():.4f}')

# ── 6c: imdb_votes_log ────────────────────────────────────────────────────────
master['imdb_votes_log'] = np.log1p(master['imdb_votes'].fillna(0)).round(4)

# ── 6d: content_soup ─────────────────────────────────────────────────────────
def build_soup(row):
    parts = []

    # Genres → normalised tokens
    genres = row['genre_list'] if isinstance(row['genre_list'], list) else []
    parts += [g.lower().replace(' ', '') for g in genres]

    # Director → single token
    director = row['director'] if isinstance(row['director'], str) else ''
    if director:
        parts.append(normalise_name(director))

    # Cast top-3 → individual tokens
    cast = row['cast_top3'] if isinstance(row['cast_top3'], list) else []
    parts += [normalise_name(name) for name in cast]

    # Keywords → individual tokens
    keywords = row['keywords_clean'] if isinstance(row['keywords_clean'], list) else []
    parts += [k.replace(' ', '') for k in keywords]

    # Overview → append as raw text
    overview = row['overview_clean'] if isinstance(row['overview_clean'], str) else ''
    if overview:
        parts.append(overview)

    return ' '.join(parts).strip()

print('Building content_soup …')
master['content_soup'] = master.apply(build_soup, axis=1)

empty_soup = (master['content_soup'] == '').sum()
print(f'  Movies with empty soup: {empty_soup:,}')
print(f'  Sample soup: {master["content_soup"].iloc[0][:120]} …')

Global mean score C = 6.1355
Bayesian score — min: 1.3277, max: 9.2999, mean: 6.2464
Building content_soup …
  Movies with empty soup: 25
  Sample soup: animation comedy family johnlasseter tomhanks timallen donrickles jealousy toy boy friendship friends rivalry boynextdoo …


---
## Step 6e — Finalise `master_movies` Schema

Select and reorder to the canonical column set defined in the preprocessing plan (NB03 §6.8).

In [8]:
MASTER_COLS = [
    'tmdb_id', 'imdb_tconst', 'title', 'original_title',
    'release_year', 'language', 'runtime_min',
    'genre_list', 'director', 'cast_top3', 'keywords_clean',
    'overview_clean', 'content_soup',
    'tmdb_score', 'tmdb_votes', 'tmdb_popularity',
    'imdb_score', 'imdb_votes', 'imdb_votes_log',
    'bayesian_score',
]

master_final = (
    master[MASTER_COLS]
    .drop_duplicates(subset='tmdb_id')  # remove duplicates from links join
    .reset_index(drop=True)
    .copy()
)

# Drop internal helper columns
master = master.drop(columns=['_score', '_votes'], errors='ignore')

print(f'master_final shape: {master_final.shape}')
print()
print('Schema:')
print(master_final.dtypes.to_string())
print()
master_final.head(3)

master_final shape: (45433, 20)

Schema:
tmdb_id              int64
imdb_tconst            str
title                  str
original_title         str
release_year         Int64
language               str
runtime_min        float64
genre_list          object
director               str
cast_top3           object
keywords_clean      object
overview_clean         str
content_soup           str
tmdb_score         float64
tmdb_votes         float64
tmdb_popularity    float64
imdb_score         float64
imdb_votes         float64
imdb_votes_log     float64
bayesian_score     float64



,tmdb_id,imdb_tconst,title,original_title,release_year,language,runtime_min,genre_list,director,cast_top3,keywords_clean,overview_clean,content_soup,tmdb_score,tmdb_votes,tmdb_popularity,imdb_score,imdb_votes,imdb_votes_log,bayesian_score
0,862,tt0114709,Toy Story,Toy Story,1995,en,81.0,"[Animation, Comedy, Family]",John Lasseter,"[Tom Hanks, Tim Allen, Don Rickles]","[jealousy, toy, boy, friendship, friends, riva...","Led by Woody, Andy's toys live happily in his ...",animation comedy family johnlasseter tomhanks ...,7.7,5415.0,21.946943,8.3,1166515.0,13.9695,8.2998
1,8844,tt0113497,Jumanji,Jumanji,1995,en,104.0,"[Adventure, Fantasy, Family]",Joe Johnston,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]","[board game, disappearance, based on children'...",When siblings Judy and Peter discover an encha...,adventure fantasy family joejohnston robinwill...,6.9,2413.0,17.015539,7.1,410488.0,12.9251,7.0998
2,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,1995,en,101.0,"[Romance, Comedy]",Howard Deutch,"[Walter Matthau, Jack Lemmon, Ann-Margret]","[fishing, best friend, old men]",A family wedding reignites the ancient feud be...,romance comedy howarddeutch waltermatthau jack...,6.5,92.0,11.712900,6.7,31569.0,10.3600,6.6982


---
## Step 7 — Time-Split Ratings

Strategy (from NB03 §6.7):
- **Train** : timestamp < `2015-01-01`  (~18M rows)
- **Val**   : `2015-01-01` ≤ timestamp < `2016-01-01`  (~3M rows)
- **Test**  : timestamp ≥ `2016-01-01`  (~5M rows)

We use the full `ratings.csv` (26M rows). Reading in chunks to avoid OOM.

In [9]:
CUT_VAL  = pd.Timestamp('2015-01-01')
CUT_TEST = pd.Timestamp('2016-01-01')
CHUNK    = 1_000_000

train_chunks, val_chunks, test_chunks = [], [], []

print(f'Reading ratings.csv in chunks of {CHUNK:,} …')
for i, chunk in enumerate(pd.read_csv(
    MOVIELENS / 'ratings.csv',
    chunksize=CHUNK,
    dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32'},
)):
    chunk['timestamp'] = pd.to_datetime(chunk['timestamp'], unit='s')

    train_chunks.append(chunk[chunk['timestamp'] <  CUT_VAL])
    val_chunks.append(  chunk[(chunk['timestamp'] >= CUT_VAL) & (chunk['timestamp'] < CUT_TEST)])
    test_chunks.append( chunk[chunk['timestamp'] >= CUT_TEST])

    if (i + 1) % 5 == 0:
        print(f'  Processed {(i+1)*CHUNK:,} rows …')

ratings_train = pd.concat(train_chunks, ignore_index=True)
ratings_val   = pd.concat(val_chunks,   ignore_index=True)
ratings_test  = pd.concat(test_chunks,  ignore_index=True)

print()
print(f'Train : {len(ratings_train):>10,} rows  '
      f'({len(ratings_train)/26_024_289*100:.1f}%)')
print(f'Val   : {len(ratings_val):>10,} rows  '
      f'({len(ratings_val)/26_024_289*100:.1f}%)')
print(f'Test  : {len(ratings_test):>10,} rows  '
      f'({len(ratings_test)/26_024_289*100:.1f}%)')

Reading ratings.csv in chunks of 1,000,000 …
  Processed 5,000,000 rows …
  Processed 10,000,000 rows …
  Processed 15,000,000 rows …
  Processed 20,000,000 rows …
  Processed 25,000,000 rows …

Train : 20,720,316 rows  (79.6%)
Val   :  1,913,720 rows  (7.4%)
Test  :  3,390,253 rows  (13.0%)


---
## Step 8 — Save All Parquet Outputs

All files written to `data_science/processed/` using Snappy compression.

In [10]:
OUTPUT_FILES = {
    'movies_clean.parquet'     : movies_clean,
    'credits_clean.parquet'    : credits_clean,
    'keywords_clean.parquet'   : keywords_clean,
    'imdb_enrichment.parquet'  : imdb_enrichment,
    'master_movies.parquet'    : master_final,
    'ratings_train.parquet'    : ratings_train,
    'ratings_val.parquet'      : ratings_val,
    'ratings_test.parquet'     : ratings_test,
}

for filename, df in OUTPUT_FILES.items():
    path = PROCESSED / filename
    df.to_parquet(path, index=False, compression='snappy')
    size_mb = path.stat().st_size / 1_048_576
    print(f'  Saved {filename:<35}  {len(df):>10,} rows  {size_mb:>7.1f} MB')

print()
print('All outputs saved.')

  Saved movies_clean.parquet                     45,433 rows     11.8 MB
  Saved credits_clean.parquet                    45,432 rows      1.4 MB
  Saved keywords_clean.parquet                   45,432 rows      0.8 MB
  Saved imdb_enrichment.parquet                  39,176 rows      0.5 MB
  Saved master_movies.parquet                    45,433 rows     26.9 MB
  Saved ratings_train.parquet                20,720,316 rows    147.4 MB
  Saved ratings_val.parquet                   1,913,720 rows     14.6 MB
  Saved ratings_test.parquet                  3,390,253 rows     25.8 MB

All outputs saved.


---
## Step 9 — Validation

Sanity checks on all Parquet outputs:
- Row counts match expectations
- No duplicate `tmdb_id` in master
- `bayesian_score` range valid
- Ratings splits are non-overlapping by timestamp
- No data leakage between train / val / test

In [11]:
print('=' * 60)
print('VALIDATION REPORT')
print('=' * 60)

# ── Reload from disk to confirm write integrity ────────────────────────────────
mm   = pd.read_parquet(PROCESSED / 'master_movies.parquet')
r_tr = pd.read_parquet(PROCESSED / 'ratings_train.parquet')
r_va = pd.read_parquet(PROCESSED / 'ratings_val.parquet')
r_te = pd.read_parquet(PROCESSED / 'ratings_test.parquet')

# ── master_movies checks ──────────────────────────────────────────────────────
dup = mm['tmdb_id'].duplicated().sum()
print(f'\nmaster_movies.parquet')
print(f'  Rows              : {len(mm):,}')
print(f'  Columns           : {mm.columns.tolist()}')
print(f'  Duplicate tmdb_id : {dup}')
print(f'  bayesian_score range: [{mm["bayesian_score"].min():.4f}, {mm["bayesian_score"].max():.4f}]')
print(f'  Empty content_soup: {(mm["content_soup"] == "").sum():,}')
print(f'  Missing genre_list: {mm["genre_list"].apply(lambda x: len(x) == 0).sum():,}')

# ── Ratings split checks ──────────────────────────────────────────────────────
total = len(r_tr) + len(r_va) + len(r_te)
print(f'\nRatings splits')
print(f'  Train : {len(r_tr):>10,}')
print(f'  Val   : {len(r_va):>10,}')
print(f'  Test  : {len(r_te):>10,}')
print(f'  Total : {total:>10,}')

print(f'\nTimestamp ranges:')
print(f'  Train : {r_tr["timestamp"].min()} → {r_tr["timestamp"].max()}')
print(f'  Val   : {r_va["timestamp"].min()} → {r_va["timestamp"].max()}')
print(f'  Test  : {r_te["timestamp"].min()} → {r_te["timestamp"].max()}')

# Overlap checks
overlap_tv = (r_tr['timestamp'] >= CUT_VAL).sum()
overlap_vt = (r_va['timestamp'] >= CUT_TEST).sum()
print(f'\nOverlap checks (should all be 0):')
print(f'  Train rows >= val cutoff  : {overlap_tv}')
print(f'  Val rows   >= test cutoff : {overlap_vt}')

print()
print('OK Validation complete.')

VALIDATION REPORT

master_movies.parquet
  Rows              : 45,433
  Columns           : ['tmdb_id', 'imdb_tconst', 'title', 'original_title', 'release_year', 'language', 'runtime_min', 'genre_list', 'director', 'cast_top3', 'keywords_clean', 'overview_clean', 'content_soup', 'tmdb_score', 'tmdb_votes', 'tmdb_popularity', 'imdb_score', 'imdb_votes', 'imdb_votes_log', 'bayesian_score']
  Duplicate tmdb_id : 0
  bayesian_score range: [1.3277, 9.2999]
  Empty content_soup: 25
  Missing genre_list: 2,442

Ratings splits
  Train : 20,720,316
  Val   :  1,913,720
  Test  :  3,390,253
  Total : 26,024,289

Timestamp ranges:
  Train : 1995-01-09 11:46:44 → 2014-12-31 23:49:53
  Val   : 2015-01-01 00:02:15 → 2015-12-31 23:59:54
  Test  : 2016-01-01 00:00:00 → 2017-08-04 06:57:50

Overlap checks (should all be 0):
  Train rows >= val cutoff  : 0
  Val rows   >= test cutoff : 0

OK Validation complete.


---
## Summary

| Output file | Rows | Description |
|---|---|---|
| `movies_clean.parquet` | ~45,460 | Cleaned TMDB metadata |
| `credits_clean.parquet` | ~45,476 | Director + top-3 cast |
| `keywords_clean.parquet` | ~46,419 | Filtered keywords |
| `imdb_enrichment.parquet` | ~38,000 | IMDB ratings + runtime |
| `master_movies.parquet` | ~45,460 | Full feature table |
| `ratings_train.parquet` | ~18M | Training ratings (< 2015) |
| `ratings_val.parquet` | ~3M | Validation ratings (2015–2015) |
| `ratings_test.parquet` | ~5M | Test ratings (≥ 2016) |

### Key fields in `master_movies.parquet`
| Field | Role |
|---|---|
| `content_soup` | Input to TF-IDF for content-based filtering |
| `bayesian_score` | Bias-corrected score for cold-start ranking |
| `genre_list` | For genre-based filtering |
| `director`, `cast_top3` | Named entity features |
| `imdb_votes_log` | Log-scaled popularity signal |

### Next → `05_content_based.ipynb`
- TF-IDF vectorisation of `content_soup`
- Cosine similarity matrix (sparse)
- Top-N recommendations by title lookup
- Optional: sentence-transformer embeddings for semantic similarity
- Precision@K evaluation